# Session 6: Pandas Fundamentals

**Course:** Python for Data Engineering  
**Phase 2:** Data Handling & Transformation

**What we'll cover:**
- DataFrames & Series
- Reading data (CSV, JSON, Excel)
- Exploring and inspecting data
- Basic column operations
- Lab: Load and explore real datasets

**Data files:** Same `data/` directory — sales, employees, transactions.

In [ ]:
import pandas as pd
import numpy as np

---

## 1. DataFrames & Series

pandas has two core data structures:

- **DataFrame** — a table (rows and columns), like a CSV loaded into memory
- **Series** — a single column from a DataFrame

Think of a DataFrame as a dict of Series, where each key is a column name.

In [ ]:
# Creating a DataFrame from a dict — common for quick testing

sales_data = {
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"],
    "quantity": [2, 10, 5, 1, 3],
    "unit_price": [999.99, 29.99, 79.50, 349.99, 59.99],
    "region": ["north", "south", "north", "east", "south"],
}

df = pd.DataFrame(sales_data)
print(type(df))
df

In [ ]:
# A Series is a single column

prices = df["unit_price"]
print(type(prices))
print(prices)

In [ ]:
# Series have built-in aggregations

print(f"Sum:    ${prices.sum():,.2f}")
print(f"Mean:   ${prices.mean():,.2f}")
print(f"Min:    ${prices.min():,.2f}")
print(f"Max:    ${prices.max():,.2f}")
print(f"Count:  {prices.count()}")

**Try it:** Create a DataFrame from the employee data below and find the average salary.

```python
employees = {
    "name": ["John Doe", "Jane Smith", "Bob Wilson", "Sarah Connor", "Mike Ross"],
    "department": ["engineering", "data", "hr", "engineering", "data"],
    "salary": [85000, 92000, 65000, 95000, 88000],
    "age": [32, 28, 45, 38, 30],
}
```

In [ ]:
# Your code here

---

## 2. Reading Data

pandas can read almost any format. These are the ones you'll use most in DE.

| Function | Format |
|----------|--------|
| `pd.read_csv()` | CSV files |
| `pd.read_json()` | JSON files |
| `pd.read_excel()` | Excel files (needs `openpyxl`) |
| `pd.read_sql()` | SQL query results |
| `pd.read_parquet()` | Parquet files (common in big data) |

In [ ]:
# Reading CSV — this replaces all the csv.DictReader code from Sessions 2-4

sales = pd.read_csv("data/sales.csv")
sales

In [ ]:
# Reading JSON — replaces json.load()

transactions = pd.read_json("data/transactions.json")
transactions

In [ ]:
# Reading employees

employees = pd.read_csv("data/employees.csv")
employees

### Useful read_csv parameters

| Parameter | What it does | Example |
|-----------|-------------|--------|
| `sep` | Column delimiter | `sep="\t"` for TSV |
| `header` | Row to use as header | `header=None` if no header |
| `names` | Column names to use | `names=["a", "b", "c"]` |
| `usecols` | Only read specific columns | `usecols=["product", "quantity"]` |
| `dtype` | Force column types | `dtype={"zip": str}` |
| `na_values` | Treat these as NaN | `na_values=["N/A", "missing"]` |
| `nrows` | Only read first N rows | `nrows=100` for preview |

In [ ]:
# Reading only specific columns — saves memory on large files

sales_slim = pd.read_csv("data/sales.csv", usecols=["product", "quantity", "region"])
sales_slim

---

## 3. Exploring Data

First thing you do after loading: understand what you have. These methods give you the full picture fast.

In [ ]:
# shape — rows x columns

print(f"Shape: {sales.shape}")
print(f"Rows:  {sales.shape[0]}")
print(f"Cols:  {sales.shape[1]}")

In [ ]:
# head() and tail() — preview rows

print("First 3 rows:")
print(sales.head(3))
print("\nLast 2 rows:")
print(sales.tail(2))

In [ ]:
# info() — column names, types, non-null counts
# this is your go-to for understanding the schema

sales.info()

In [ ]:
# dtypes — just the types

print(sales.dtypes)

In [ ]:
# columns — list of column names

print(f"Columns: {list(sales.columns)}")

In [ ]:
# describe() — stats for numeric columns

sales.describe()

In [ ]:
# value_counts() — frequency of values in a column
# great for checking data quality

print("Products:")
print(sales["product"].value_counts())
print("\nRegions:")
print(sales["region"].value_counts())

In [ ]:
# unique() and nunique() — distinct values

print(f"Unique regions: {sales['region'].unique()}")
print(f"Number of unique products: {sales['product'].nunique()}")

### Quick reference: exploration methods

| Method | What it shows |
|--------|---------------|
| `df.shape` | (rows, columns) |
| `df.head(n)` | First n rows |
| `df.tail(n)` | Last n rows |
| `df.info()` | Schema: column names, types, nulls |
| `df.dtypes` | Just the column types |
| `df.describe()` | Stats for numeric columns |
| `df.columns` | Column names |
| `df[col].value_counts()` | Frequency table |
| `df[col].unique()` | Distinct values |
| `df[col].nunique()` | Number of distinct values |

**Try it:** Load `data/employees.csv` into a DataFrame and answer:
1. How many rows and columns?
2. What are the column types?
3. How many unique departments?
4. What's the salary range (min and max)?

In [ ]:
# Your code here

---

## 4. Selecting Data

Three ways to grab data from a DataFrame.

In [ ]:
# Single column — returns a Series

products = sales["product"]
print(products)

In [ ]:
# Multiple columns — returns a DataFrame

subset = sales[["product", "quantity", "region"]]
subset

In [ ]:
# Row selection with iloc (by position) and loc (by label/condition)

# First row
print("First row:")
print(sales.iloc[0])

# Rows 1-3, columns 0-2
print("\nSlice:")
print(sales.iloc[1:4, 0:3])

---

## 5. Adding and Modifying Columns

In data pipelines, you constantly create new columns from existing ones.

In [ ]:
# Start fresh

df = pd.read_csv("data/sales.csv")

# Clean the price column — remove $ and convert to float
df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)

# Add a computed column
df["total"] = df["quantity"] * df["unit_price"]

# Clean string columns
df["product"] = df["product"].str.strip().str.title()
df["region"] = df["region"].str.strip().str.lower()

df

In [ ]:
# Renaming columns

df_renamed = df.rename(columns={"unit_price": "price", "quantity": "qty"})
print(df_renamed.columns.tolist())

In [ ]:
# Dropping columns you don't need

df_slim = df.drop(columns=["date"])
df_slim

---

## 6. Saving Data

Write DataFrames back to files — closing the loop on your pipeline.

In [ ]:
# Save to CSV
df.to_csv("data/sales_pandas_clean.csv", index=False)

# Save to JSON
df.to_json("data/sales_pandas_clean.json", orient="records", indent=2)

print("Saved CSV and JSON")

# Verify by reading back
check = pd.read_csv("data/sales_pandas_clean.csv")
print(f"\nCSV has {len(check)} rows")
check.head(3)

---

## Lab Exercises

---

### Lab 1: Load, Explore, Report

Load `data/sales_messy.csv` (the file with bad data from Session 4) and write an exploration report.

**Steps:**
1. Load the file with `pd.read_csv()`
2. Check `shape`, `info()`, `dtypes`
3. Use `value_counts()` on each column to spot bad data
4. Print a summary: how many rows, columns, nulls per column, unique values per column

The goal is **understanding the data before cleaning** — this is what you do first on every new dataset.

In [ ]:
import pandas as pd

# Your code here

---

### Lab 2: Build a pandas ETL Mini-Pipeline

Recreate the ETL pipeline from Session 3 Lab 3, but with pandas instead of manual CSV/JSON code.

**Steps:**
1. **Extract**: Load `data/sales.csv` with `pd.read_csv()`
2. **Transform**:
   - Clean `unit_price` (remove `$`, convert to float)
   - Strip and normalize `product` (title case) and `region` (lowercase)
   - Add a `total` column (quantity × unit_price)
   - Filter out rows where quantity ≤ 0
3. **Load**: Save clean data to `data/sales_etl_output.csv`
4. Print: total rows processed, total revenue, revenue by region

In [ ]:
import pandas as pd

# Your code here

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| DataFrame | A table — rows and columns, like a CSV in memory |
| Series | A single column from a DataFrame |
| Reading data | `pd.read_csv()`, `pd.read_json()` — one line to load |
| Exploring | `shape`, `info()`, `describe()`, `value_counts()` — understand before you transform |
| Selecting | `df["col"]` for a column, `df[["a", "b"]]` for multiple, `iloc`/`loc` for rows |
| Modifying | `df["new"] = ...` to add, `.str` methods for strings, `.astype()` for types |
| Saving | `to_csv()`, `to_json()` — always set `index=False` |

**Key patterns:**
- Always explore data first (`info()`, `value_counts()`) before transforming
- pandas replaces 90% of the manual loop + dict work from Phase 1
- Use `str` accessor for string operations on columns
- Set `index=False` when saving to CSV (avoids extra unnamed column)

**Next session:** Data Cleaning with Pandas — handling nulls, filtering, sorting, and transforming messy data.